# Upper-Intermediate — AI 기반 지반함몰 위험도 예측 실습

각 단계의 Task와 코드 골격이 제공됩니다. `None`을 알맞은 값으로 바꾸어 핵심 학습·평가 로직을 완성하세요.

공통 조건: `random_state=42`, 테스트 데이터 25%, 위험등급 비율 유지

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from xgboost import XGBClassifier

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")

In [ ]:
candidates = [
    Path("../data/hannam_sinkhole_data.xlsx"),
    Path("data/hannam_sinkhole_data.xlsx"),
    Path("hannam_sinkhole_data.xlsx"),
]
DATA_PATH = next((p for p in candidates if p.exists()), None)
DATA_URL = (
    "https://raw.githubusercontent.com/GeoSeonghoHong/"
    "Seongho-Hong/main/data/hannam_sinkhole_data.xlsx"
)

if DATA_PATH is not None:
    print("로컬 데이터 사용:", DATA_PATH)
    df = pd.read_excel(DATA_PATH, sheet_name="training_data")
else:
    print("로컬 파일이 없어 GitHub에서 데이터를 불러옵니다.")
    df = pd.read_excel(DATA_URL, sheet_name="training_data")

print("shape:", df.shape)
df.head()

## Task 1. 데이터 구조와 위험등급 분포 확인

1. 행과 열의 개수, 자료형을 확인하세요.
2. 결측치와 중복 행의 개수를 확인하세요.
3. Low/Medium/High 등급별 개수를 확인하세요.

힌트: `df.info()`, `isna()`, `duplicated()`, `value_counts()`

In [ ]:
print(df.info())
print("결측치:", int(df.isna().sum().sum()))
print("중복 행:", int(df.duplicated().sum()))
display(df["risk"].value_counts().sort_index())

## Task 2. 세 가지 입력 변수 집합 설계

1. Model A에는 관로 밀집도 변수 2개를 사용하세요.
2. Model B에는 시공 시기별 관로 길이, 밀집도, 공동 개수를 사용하세요.
3. Model C에는 Model B의 변수와 파생 변수를 함께 사용하세요.

힌트: `GUIDE.md`의 주요 변수와 파생 변수 표를 참고하세요.

In [ ]:
water_features = [f"water_Y_{year}" for year in range(5, 60, 5)]
sewer_features = [f"sewer_Y_{year}" for year in range(5, 25, 5)]

# TODO 1: 하수도관 밀집도 컬럼명을 입력하세요.
basic_features = ["water_Density", None]

# TODO 2: 공동 개수 컬럼명을 입력하세요.
raw_features = water_features + ["water_Density"] + sewer_features + [
    "sewer_Density", None
]

derived_features = [
    "total_water_length", "total_sewer_length", "pipe_total_density",
    "water_sewer_ratio", "old_pipe_ratio", "recent_pipe_ratio",
    "cavity_density_interaction",
]
engineered_features = raw_features + derived_features

## Task 3. 공통 학습·평가 함수 작성

1. 데이터를 75:25로 분리하세요.
2. `stratify`를 사용하여 위험등급 비율을 유지하세요.
3. XGBoost 모델을 학습하세요.
4. 평가 데이터를 예측하세요.
5. Accuracy와 Macro F1-score를 계산하세요.

힌트: `test_size=0.25`, `stratify=y`, `fit`, `predict`, `average="macro"`

In [ ]:
def evaluate_model(name, features):
    X = df[features].copy()
    y = df["risk"].copy()

    # TODO 3: None을 알맞은 값으로 바꾸세요.
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=None,
        random_state=RANDOM_STATE,
        stratify=None,
    )

    model = XGBClassifier(
        n_estimators=200,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        eval_metric="mlogloss",
        random_state=RANDOM_STATE,
    )

    # TODO 4: 학습 데이터와 평가 데이터를 전달하세요.
    model.fit(None, None)
    pred = model.predict(None)

    # TODO 5: 정답과 예측값, 평균 방식을 입력하세요.
    accuracy = accuracy_score(None, None)
    macro_f1 = f1_score(None, None, average=None)

    return {
        "model": name,
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "model_object": model,
        "features": features,
        "y_test": y_test,
        "pred": pred,
    }

## Task 4. Model A/B/C 학습과 성능 비교

세 모델을 같은 조건으로 학습하고 Accuracy와 Macro F1-score를 표와 그래프로 비교하세요.

힌트: 각 모델에 `basic_features`, `raw_features`, `engineered_features`를 순서대로 전달합니다.

In [ ]:
# TODO 6: 각 모델에 사용할 변수 목록을 입력하세요.
result_a = evaluate_model("Model A: density only", None)
result_b = evaluate_model("Model B: raw features", None)
result_c = evaluate_model("Model C: engineered features", None)

results = [result_a, result_b, result_c]
comparison = pd.DataFrame([
    {"model": r["model"], "accuracy": r["accuracy"], "macro_f1": r["macro_f1"]}
    for r in results
]).set_index("model")
display(comparison.round(3))
comparison.plot(kind="bar", ylim=(0, 1), figsize=(9, 4))
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()

## Task 5. 오분류 분석

Macro F1-score가 가장 높은 모델을 선택하고 Confusion Matrix와 classification report를 출력하세요.

질문: 실제 High 등급을 놓친 사례는 몇 개인가요?

힌트: `max(results, key=...)`와 `confusion_matrix()`를 사용하세요.

In [ ]:
# TODO 7: Macro F1-score를 기준으로 최고 모델을 선택하세요.
best_result = max(results, key=lambda r: None)
cm = confusion_matrix(best_result["y_test"], best_result["pred"])
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Low", "Medium", "High"],
            yticklabels=["Low", "Medium", "High"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

## Task 6. 변수 중요도와 공학적 해석

Model C의 변수 중요도 상위 15개를 시각화하고 다음 내용을 3~5문장으로 정리하세요.

1. 가장 중요한 변수
2. Model A/B/C의 성능 차이
3. 지반함몰 위험과 관련될 수 있는 공학적 이유
4. 변수 중요도를 인과관계로 해석하면 안 되는 이유

In [ ]:
# TODO 8: Model C의 변수명과 중요도를 이용해 표를 완성하세요.
importance = pd.DataFrame({
    "feature": None,
    "importance": None,
}).sort_values("importance", ascending=False)

display(importance.head(15))
importance.head(15).sort_values("importance").plot(
    kind="barh", x="feature", y="importance", figsize=(8, 6), legend=False
)
plt.tight_layout()
plt.show()